# 01 — EDA (Phase 1, exploration only)

Inspect class balance, feature ranges, and the time axis / drift periods of the
ingested dataset. **No production code lives here** — the runnable pipeline is in
`src/mlops_drift/`. Run `make data` (or `uv run python -m mlops_drift.data.ingest`)
first to produce `data/raw/dataset.parquet`.

In [ ]:
import pandas as pd
from mlops_drift.config import get_config
from mlops_drift.data import features as feat

cfg = get_config()
df = pd.read_parquet(cfg.raw_dir / 'dataset.parquet')
feature_cols = feat.select_feature_cols(df)
print(df.shape, '| features:', feature_cols)
df.head()

## Class balance (overall and per period)
Imbalanced binary target — report F1 / PR-AUC primarily, not accuracy.

In [ ]:
print('overall attack rate:', round(df['label'].mean(), 3))
df.groupby('period')['label'].agg(['count', 'mean'])

## Feature ranges and reference-vs-drift shift
Some features shift mean/variance in the drift period — the covariate drift the
monitor detects label-free.

In [ ]:
ref = df[df['period'] == 'reference'][feature_cols].mean()
drift = df[df['period'] == 'drift'][feature_cols].mean()
shift = (drift - ref).abs().sort_values(ascending=False)
shift.to_frame('abs_mean_shift')

## Time axis
`t` is a strictly increasing integer; all reference rows precede all drift rows.
Splits must be time-aware (train earlier, evaluate later) — never shuffle across.

In [ ]:
print('t monotonic increasing:', df['t'].is_monotonic_increasing)
print('max reference t:', df.loc[df.period=='reference','t'].max())
print('min drift t:    ', df.loc[df.period=='drift','t'].min())